# LeetCode 1051 — Height Checker bằng Python

**Link đề:** https://leetcode.com/problems/height-checker/
**Độ khó:** Easy · **Chủ đề:** Array, Sorting, Counting Sort

```python
class Solution:
    def heightChecker(self, heights: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Đọc đúng đề: đếm **số vị trí** mà học sinh đứng sai so với hàng đã sắp xếp — không phải "số lần đổi chỗ".
2. Phân biệt `sorted(list)` (tạo list mới) và `list.sort()` (sắp tại chỗ) — lỗi kinh điển của người mới.
3. Dùng `zip`, `enumerate`, `sum` trên giá trị boolean để viết code gọn.
4. Hiểu **Counting Sort** (sắp xếp đếm) khi giá trị nằm trong miền nhỏ `1..100` → `O(n + K)`.
5. Trace hai lời giải, chứng minh tính đúng, so sánh Time/Space, tự làm các biến thể.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — có bao nhiêu học sinh đứng sai chỗ?

Trường chụp ảnh, học sinh cần đứng một hàng theo chiều cao **không giảm** (non-decreasing: người sau cao bằng hoặc hơn người trước). Gọi `expected` là thứ tự đúng đó.

Cho mảng `heights` là thứ tự **hiện tại**. Trả về **số chỉ số `i`** mà `heights[i] != expected[i]`.

| `heights` | `expected` | Output | Giải thích |
|---|---|:---:|---|
| `[1,1,4,2,1,3]` | `[1,1,1,2,3,4]` | `3` | lệch ở chỉ số 2, 4, 5 |
| `[5,1,2,3,4]` | `[1,2,3,4,5]` | `5` | lệch mọi vị trí |
| `[1,2,3,4,5]` | `[1,2,3,4,5]` | `0` | đã đúng thứ tự |

**Constraints:**
- `1 <= heights.length <= 100`
- `1 <= heights[i] <= 100`

Câu hỏi đặt ra:
- "Thứ tự đúng" `expected` lấy ở đâu? → chính là `heights` **sau khi sắp xếp**.
- Sắp xếp xong rồi làm gì? → so từng vị trí, đếm chỗ khác.
- Miền giá trị chỉ `1..100`: có tận dụng được không? → Counting Sort.

In [1]:
heights = [1, 1, 4, 2, 1, 3]
expected = [1, 1, 1, 2, 3, 4]
for i in range(len(heights)):
    mark = "LỆCH" if heights[i] != expected[i] else "ok"
    print(f"i={i}: heights={heights[i]}, expected={expected[i]} -> {mark}")

i=0: heights=1, expected=1 -> ok
i=1: heights=1, expected=1 -> ok
i=2: heights=4, expected=1 -> LỆCH
i=3: heights=2, expected=2 -> ok
i=4: heights=1, expected=3 -> LỆCH
i=5: heights=3, expected=4 -> LỆCH


## 2. Nền tảng Python cần biết

### 2.1. `sorted(x)` khác `x.sort()` như thế nào?

| | `sorted(x)` | `x.sort()` |
|---|---|---|
| Kiểu | hàm dựng sẵn | phương thức của list |
| Trả về | **list mới** đã sắp xếp | `None` |
| List gốc | **giữ nguyên** | **bị thay đổi** |

Bài này cần **cả hai** mảng (gốc và đã sắp) để so sánh → dùng `sorted(heights)`.

In [2]:
a = [3, 1, 2]
b_sorted = sorted(a)
print("a gốc:", a, "| sorted(a):", b_sorted)   # a vẫn [3, 1, 2]

c = [3, 1, 2]
ret = c.sort()
print("c sau sort():", c, "| giá trị trả về:", ret)   # c đã [1, 2, 3], ret là None

a gốc: [3, 1, 2] | sorted(a): [1, 2, 3]
c sau sort(): [1, 2, 3] | giá trị trả về: None


### 2.2. Sắp xếp không giảm với giá trị trùng

`sorted` sắp theo thứ tự tăng dần, giá trị bằng nhau đứng cạnh nhau. Đó đúng là "non-decreasing". Python dùng thuật toán **Timsort**, `O(n log n)`.

In [3]:
print(sorted([1, 1, 4, 2, 1, 3]))   # [1, 1, 1, 2, 3, 4]
print(sorted([5, 1, 2, 3, 4]))      # [1, 2, 3, 4, 5]
print(sorted([2, 2, 2]))            # [2, 2, 2]

[1, 1, 1, 2, 3, 4]
[1, 2, 3, 4, 5]
[2, 2, 2]


### 2.3. `zip(a, b)` — đi song song hai list

`zip` ghép phần tử cùng vị trí thành cặp `(a[0], b[0]), (a[1], b[1]), ...`. Tránh phải viết `a[i]`, `b[i]`.

In [4]:
a = [1, 1, 4]
b2 = [1, 1, 1]
for x, y in zip(a, b2):
    print(x, y, "khác" if x != y else "giống")
print(list(zip(a, b2)))

1 1 giống
1 1 giống
4 1 khác
[(1, 1), (1, 1), (4, 1)]


### 2.4. `True`/`False` cũng là số: `sum` trên điều kiện

Trong Python `True == 1` và `False == 0`. Vì vậy `sum(điều_kiện for ...)` đếm số lần điều kiện đúng.

`(x != y for x, y in zip(a, b))` là **generator expression** — giống list comprehension nhưng không tạo list, sinh từng giá trị khi `sum` cần.

In [5]:
print(True + True + False)                  # 2
flags = [1 != 1, 1 != 1, 4 != 1]
print(flags, "-> sum =", sum(flags))        # [False, False, True] -> 1
a = [1, 1, 4, 2, 1, 3]
e = sorted(a)
print(sum(x != y for x, y in zip(a, e)))    # 3

2
[False, False, True] -> sum = 1
3


## 3. Cách 1 — Trực giác: sắp xếp bản sao rồi so từng vị trí

Đúng như gợi ý của đề:

1. `expected = sorted(heights)`.
2. Duyệt `i` từ `0` đến `n - 1`, nếu `heights[i] != expected[i]` thì `count += 1`.
3. Trả về `count`.

In [6]:
def height_checker_sort(heights: list[int]) -> int:
    expected = sorted(heights)        # bản sao đã sắp, heights giữ nguyên
    count = 0
    for i in range(len(heights)):
        if heights[i] != expected[i]:
            count += 1
    return count

print(height_checker_sort([1, 1, 4, 2, 1, 3]))   # 3
print(height_checker_sort([5, 1, 2, 3, 4]))      # 5
print(height_checker_sort([1, 2, 3, 4, 5]))      # 0

3
5
0


### Viết gọn bằng `zip` + `sum`

Cùng ý tưởng, chỉ một dòng. Hiểu bản dài trước rồi mới dùng bản ngắn.

In [7]:
def height_checker_zip(heights: list[int]) -> int:
    return sum(h != e for h, e in zip(heights, sorted(heights)))

print(height_checker_zip([1, 1, 4, 2, 1, 3]))
print(height_checker_zip([5, 1, 2, 3, 4]))

3
5


## 4. Trace cách 1 với `heights = [1,1,4,2,1,3]`

| `i` | `heights[i]` | `expected[i]` | Khác? | `count` sau |
|---:|:---:|:---:|:---:|:---:|
| 0 | 1 | 1 | không | 0 |
| 1 | 1 | 1 | không | 0 |
| 2 | 4 | 1 | **có** | 1 |
| 3 | 2 | 2 | không | 1 |
| 4 | 1 | 3 | **có** | 2 |
| 5 | 3 | 4 | **có** | 3 |

In [8]:
def trace_height_checker_sort(heights: list[int]) -> int:
    expected = sorted(heights)
    print("heights :", heights)
    print("expected:", expected)
    count = 0
    for i, (h, e) in enumerate(zip(heights, expected)):
        if h != e:
            count += 1
        print(f"i={i}: h={h}, e={e}, khác={h != e!s:5}, count={count}")
    return count

trace_height_checker_sort([1, 1, 4, 2, 1, 3])

heights : [1, 1, 4, 2, 1, 3]
expected: [1, 1, 1, 2, 3, 4]
i=0: h=1, e=1, khác=False, count=0
i=1: h=1, e=1, khác=False, count=0
i=2: h=4, e=1, khác=True , count=1
i=3: h=2, e=2, khác=False, count=1
i=4: h=1, e=3, khác=True , count=2
i=5: h=3, e=4, khác=True , count=3


3

### Cú pháp `for i, (h, e) in enumerate(zip(...))`

- `zip(heights, expected)` cho cặp `(h, e)`.
- `enumerate(...)` bọc thêm chỉ số: `(i, (h, e))`.
- Viết `i, (h, e)` để **tách lồng** (nested unpacking) ngay trong vòng `for`.

In [9]:
for item in enumerate(zip("ab", "xy")):
    print("nguyên cặp:", item)
for i, (p, q) in enumerate(zip("ab", "xy")):
    print("đã tách:", i, p, q)

nguyên cặp: (0, ('a', 'x'))
nguyên cặp: (1, ('b', 'y'))
đã tách: 0 a x
đã tách: 1 b y


## 5. Nút thắt & insight: có cần sắp xếp so sánh `O(n log n)` không?

Với `n <= 100` thì `sorted` đã quá nhanh. Nhưng đề có tag **Counting Sort** — một ý tưởng đáng học:

- Chiều cao chỉ nằm trong `1..100` (miền nhỏ, gọi `K = 100`).
- Thay vì so sánh các phần tử, ta **đếm** mỗi chiều cao xuất hiện bao nhiêu lần: `freq[h]`.
- Rồi "trải" lại theo thứ tự `h = 1, 2, ..., 100`: chiều cao `h` lặp `freq[h]` lần → ra đúng `expected`.

Không có phép so sánh nào giữa hai học sinh! Chi phí `O(n + K)`.

Ví dụ `[1,1,4,2,1,3]`: `freq[1]=3, freq[2]=1, freq[3]=1, freq[4]=1` → trải ra `1,1,1,2,3,4`.

In [10]:
heights = [1, 1, 4, 2, 1, 3]
freq = [0] * 101                 # chỉ số 0..100; dùng 1..100
for h in heights:
    freq[h] += 1
print("freq[1..5] =", freq[1:6])

expected = []
for h in range(1, 101):
    expected.extend([h] * freq[h])   # [h] * k tạo list có k phần tử h
print("expected  =", expected)

freq[1..5] = [3, 1, 1, 1, 0]
expected  = [1, 1, 1, 2, 3, 4]


### Giải thích `[0] * 101` và `[h] * k`

- `[0] * 101` tạo list 101 số 0 — bảng đếm, chỉ số chính là chiều cao.
- `[h] * k` tạo list gồm `k` phần tử `h`; `k = 0` cho list rỗng.
- `lst.extend(other)` nối **từng phần tử** của `other` vào cuối `lst` (khác `append` thêm cả list như 1 phần tử).

In [11]:
print([7] * 3, [7] * 0)
x = [1]
x.extend([2, 2])
print("extend:", x)
y = [1]
y.append([2, 2])
print("append:", y)

[7, 7, 7] []
extend: [1, 2, 2]
append: [1, [2, 2]]


## 6. Cách 2 (lời giải chính) — Counting Sort, so sánh không cần dựng `expected`

Ta còn có thể **không dựng** mảng `expected`: dùng con trỏ `cur` = chiều cao nhỏ nhất chưa dùng hết trong `freq`.

**Đoạn 1 — đếm:** `freq[h] += 1` cho mỗi `h`.

**Đoạn 2 — duyệt `heights` theo thứ tự:** tại vị trí `i`, chiều cao đúng phải là `cur` (bỏ qua các `cur` có `freq[cur] == 0`).

**Đoạn 3 — so sánh và "dùng" một suất:** nếu `heights[i] != cur` thì `count += 1`; sau đó `freq[cur] -= 1`.

In [12]:
def height_checker_counting(heights: list[int]) -> int:
    freq = [0] * 101
    for h in heights:                 # Đoạn 1
        freq[h] += 1

    count = 0
    cur = 1                           # chiều cao nhỏ nhất còn "suất"
    for h in heights:                 # Đoạn 2
        while freq[cur] == 0:         # bỏ chiều cao đã dùng hết
            cur += 1
        if h != cur:                  # Đoạn 3
            count += 1
        freq[cur] -= 1
    return count

print(height_checker_counting([1, 1, 4, 2, 1, 3]))   # 3
print(height_checker_counting([5, 1, 2, 3, 4]))      # 5
print(height_checker_counting([1, 2, 3, 4, 5]))      # 0

3
5
0


## 7. Trace Counting Sort với `[1,1,4,2,1,3]`

Sau đoạn 1: `freq = {1:3, 2:1, 3:1, 4:1}`.

| `i` | `h` | `cur` (đúng ra) | Khác? | `count` | `freq` sau khi dùng |
|---:|:---:|:---:|:---:|:---:|---|
| 0 | 1 | 1 | không | 0 | `{1:2,2:1,3:1,4:1}` |
| 1 | 1 | 1 | không | 0 | `{1:1,...}` |
| 2 | 4 | 1 | **có** | 1 | `{1:0,...}` |
| 3 | 2 | 2 (bỏ qua 1) | không | 1 | `{2:0,...}` |
| 4 | 1 | 3 | **có** | 2 | `{3:0,...}` |
| 5 | 3 | 4 | **có** | 3 | `{4:0}` |

In [13]:
def trace_height_checker_counting(heights: list[int]) -> int:
    freq = [0] * 101
    for h in heights:
        freq[h] += 1
    print("freq (khác 0):", {v: freq[v] for v in range(101) if freq[v]})
    count, cur = 0, 1
    for i, h in enumerate(heights):
        while freq[cur] == 0:
            cur += 1
        if h != cur:
            count += 1
        freq[cur] -= 1
        alive = {v: freq[v] for v in range(101) if freq[v]}
        print(f"i={i}: h={h}, đúng ra={cur}, khác={h != cur!s:5}, count={count}, còn={alive}")
    return count

trace_height_checker_counting([1, 1, 4, 2, 1, 3])

freq (khác 0): {1: 3, 2: 1, 3: 1, 4: 1}
i=0: h=1, đúng ra=1, khác=False, count=0, còn={1: 2, 2: 1, 3: 1, 4: 1}
i=1: h=1, đúng ra=1, khác=False, count=0, còn={1: 1, 2: 1, 3: 1, 4: 1}
i=2: h=4, đúng ra=1, khác=True , count=1, còn={2: 1, 3: 1, 4: 1}
i=3: h=2, đúng ra=2, khác=False, count=1, còn={3: 1, 4: 1}
i=4: h=1, đúng ra=3, khác=True , count=2, còn={4: 1}
i=5: h=3, đúng ra=4, khác=True , count=3, còn={}


3

### Dict comprehension trong hàm trace

`{v: freq[v] for v in range(101) if freq[v]}` tạo **dict** chỉ gồm các chiều cao còn đếm > 0 — để in gọn. Cú pháp: `{key: value for ... if ...}`.

In [14]:
freq_demo = [0, 2, 0, 1]
print({v: freq_demo[v] for v in range(len(freq_demo)) if freq_demo[v]})   # {1: 2, 3: 1}

{1: 2, 3: 1}


## 8. Cách 3 — `collections.Counter` (bản dễ đọc của Counting Sort)

`Counter(heights)` là dict đếm có sẵn trong thư viện chuẩn. Dựng `expected` bằng cách duyệt các key **đã sắp** — chỉ có tối đa 100 key khác nhau, nên sắp key rất rẻ.

In [15]:
from collections import Counter

cnt = Counter([1, 1, 4, 2, 1, 3])
print(cnt)                       # Counter({1: 3, 4: 1, 2: 1, 3: 1})
print(sorted(cnt))               # key đã sắp: [1, 2, 3, 4]

def height_checker_counter(heights: list[int]) -> int:
    cnt = Counter(heights)
    expected = []
    for h in sorted(cnt):
        expected += [h] * cnt[h]
    return sum(a != b for a, b in zip(heights, expected))

print(height_checker_counter([1, 1, 4, 2, 1, 3]))

Counter({1: 3, 4: 1, 2: 1, 3: 1})
[1, 2, 3, 4]
3


## 9. Code nộp LeetCode

### Bản sắp xếp (ngắn, dễ nhớ)

In [16]:
class Solution:
    def heightChecker(self, heights: list[int]) -> int:
        expected = sorted(heights)
        return sum(h != e for h, e in zip(heights, expected))

assert Solution().heightChecker([1, 1, 4, 2, 1, 3]) == 3
assert Solution().heightChecker([5, 1, 2, 3, 4]) == 5
assert Solution().heightChecker([1, 2, 3, 4, 5]) == 0
print("Solution (sort) PASS")

Solution (sort) PASS


### Bản Counting Sort `O(n + K)`

In [17]:
class SolutionCounting:
    def heightChecker(self, heights: list[int]) -> int:
        freq = [0] * 101
        for h in heights:
            freq[h] += 1
        count, cur = 0, 1
        for h in heights:
            while freq[cur] == 0:
                cur += 1
            if h != cur:
                count += 1
            freq[cur] -= 1
        return count

assert SolutionCounting().heightChecker([1, 1, 4, 2, 1, 3]) == 3
assert SolutionCounting().heightChecker([5, 1, 2, 3, 4]) == 5
assert SolutionCounting().heightChecker([1, 2, 3, 4, 5]) == 0
print("SolutionCounting PASS")

SolutionCounting PASS


## 10. Vì sao đúng?

**Cách 1:** `expected` theo định nghĩa của đề chính là `heights` sắp không giảm. Ta so **đúng từng vị trí** và đếm chỗ khác — trực tiếp là định nghĩa đáp án. Lưu ý: dù có nhiều học sinh cùng chiều cao, mảng `expected` (dãy số) vẫn **duy nhất**, nên đáp án xác định.

**Cách 2 (invariant):** Trước khi xử lý vị trí `i`, mảng `freq` lưu **số suất còn lại** của mỗi chiều cao trong phần `expected[i:]`, và `cur` là chiều cao nhỏ nhất còn suất.
- Vì `expected` không giảm, `expected[i]` chính là chiều cao nhỏ nhất còn suất → bằng `cur` (sau vòng `while`).
- So `heights[i]` với `cur` = so với `expected[i]`. Trừ một suất `freq[cur] -= 1` để giữ invariant cho `i + 1`.

`cur` không bao giờ vượt quá 100 khi còn phần tử chưa xử lý, vì tổng suất còn lại = số phần tử còn lại > 0.

## 11. Độ phức tạp

Gọi `n = len(heights)`, `K = 100` là miền giá trị.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. `sorted` + so sánh | `O(n log n)` | `O(n)` (mảng `expected`) | ngắn, phổ quát |
| 2. Counting Sort + con trỏ `cur` | `O(n + K)` | `O(K)` | `cur` chỉ tăng, tổng ≤ `K` bước |
| 3. `Counter` + sắp key | `O(n + d log d)`, `d` ≤ `K` số giá trị khác nhau | `O(n)` | dễ đọc |

Ở cách 2, vòng `while` nằm trong `for` nhưng **không** làm `O(n·K)`: `cur` chỉ tăng, tổng số lần tăng ≤ 100 trong cả hàm.

## 12. Bộ test

Nhóm test:
1. Ba ví dụ của đề.
2. Một phần tử; toàn bộ giống nhau.
3. Mảng giảm dần (độ dài lẻ: phần tử giữa đứng đúng chỗ!).
4. Giá trị biên `1` và `100`.
5. Đối chiếu ngẫu nhiên 4 cách.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:26} PASS -> {actual}")

tests = [
    ("đề 1", [1, 1, 4, 2, 1, 3], 3),
    ("đề 2", [5, 1, 2, 3, 4], 5),
    ("đề 3", [1, 2, 3, 4, 5], 0),
    ("một phần tử", [7], 0),
    ("toàn giống nhau", [2, 2, 2, 2], 0),
    ("giảm dần lẻ (giữa đúng)", [5, 4, 3, 2, 1], 4),
    ("giảm dần chẵn", [4, 3, 2, 1], 4),
    ("biên 1 và 100", [100, 1], 2),
    ("đổi chỗ 2 phần tử", [1, 3, 2, 4], 2),
]
for name, h, exp in tests:
    check(name, height_checker_sort(h), exp)
    assert height_checker_counting(h) == exp

đề 1                       PASS -> 3
đề 2                       PASS -> 5
đề 3                       PASS -> 0
một phần tử                PASS -> 0
toàn giống nhau            PASS -> 0
giảm dần lẻ (giữa đúng)    PASS -> 4
giảm dần chẵn              PASS -> 4
biên 1 và 100              PASS -> 2
đổi chỗ 2 phần tử          PASS -> 2


In [19]:
import random
random.seed(1051)
for _ in range(3000):
    n = random.randint(1, 100)
    h = [random.randint(1, 100) for _ in range(n)]
    original = h[:]
    r1 = height_checker_sort(h)
    r2 = height_checker_zip(h)
    r3 = height_checker_counting(h)
    r4 = height_checker_counter(h)
    assert r1 == r2 == r3 == r4, (h, r1, r2, r3, r4)
    assert h == original          # không cách nào được làm hỏng input
print("3000 test ngẫu nhiên: 4 cách khớp nhau, input không bị sửa")

3000 test ngẫu nhiên: 4 cách khớp nhau, input không bị sửa


## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `heights.sort()` rồi so với chính nó

```python
expected = heights
expected.sort()          # sắp luôn heights vì cùng một list!
```

`expected = heights` **không** tạo bản sao — hai tên cùng trỏ một list (aliasing). Sắp `expected` cũng là sắp `heights` → mọi vị trí bằng nhau → luôn trả về 0.

In [20]:
def wrong_alias(heights):
    expected = heights
    expected.sort()
    return sum(h != e for h, e in zip(heights, expected))

print("sai :", wrong_alias([1, 1, 4, 2, 1, 3]))      # 0
print("đúng:", height_checker_sort([1, 1, 4, 2, 1, 3]))
x = [3, 1]
y = x
print("y is x?", y is x)   # True: cùng một list, không phải bản sao

sai : 0
đúng: 3
y is x? True


**Sửa:** `expected = sorted(heights)` hoặc `expected = heights[:]` rồi `expected.sort()`.

### Lỗi 2: Gán kết quả của `.sort()`

```python
expected = heights.sort()    # expected là None!
```
→ `zip(heights, None)` báo `TypeError`.

In [21]:
h = [3, 1, 2]
expected = h.sort()
print("expected =", expected)
try:
    sum(a != b for a, b in zip(h, expected))
except TypeError as e:
    print("TypeError:", e)

expected = None
TypeError: 'NoneType' object is not iterable


### Lỗi 3: Đếm "số lần đổi chỗ tối thiểu" thay vì "số vị trí lệch"

Với `[5,1,2,3,4]` có người trả 1 (vì "chỉ cần đưa 5 về cuối") hoặc 4 (bubble sort swap). Đề hỏi **số chỉ số khác nhau** → 5.

### Lỗi 4: Bảng đếm cỡ 100 thay vì 101

`freq = [0] * 100` thì `freq[100]` báo `IndexError` vì chỉ số chạy `0..99`. Chiều cao tới 100 → cần `[0] * 101`.

In [22]:
freq_bad = [0] * 100
try:
    freq_bad[100] += 1
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


## 14. Mẹo nhận dạng pattern

- Đề có cụm "**so với thứ tự đã sắp xếp**" → nghĩ ngay `sorted(...)` rồi so vị trí.
- Giá trị **nằm trong miền nhỏ** (`0..100`, `a..z`, `0..9`) → có thể thay sắp xếp bằng **đếm** (counting sort, bucket).
- "Đếm số phần tử thỏa điều kiện" → `sum(cond for ...)`.

| Bài liên quan | Ý tưởng |
|---|---|
| 1365. How Many Numbers Are Smaller Than the Current Number | đếm + cộng dồn trên miền `0..100` |
| 75. Sort Colors | đếm 3 giá trị |
| 581. Shortest Unsorted Continuous Subarray | so sánh với bản sắp, tìm đoạn lệch |
| 1122. Relative Sort Array | counting sort theo thứ tự tùy biến |

## 15. Biến thể

### Biến thể A — Trả về danh sách chỉ số lệch (không chỉ số lượng)

In [23]:
def mismatched_indices(heights: list[int]) -> list[int]:
    expected = sorted(heights)
    return [i for i, (h, e) in enumerate(zip(heights, expected)) if h != e]

print(mismatched_indices([1, 1, 4, 2, 1, 3]))   # [2, 4, 5]

[2, 4, 5]


### Biến thể B — Đoạn liên tiếp ngắn nhất cần sắp lại (ý LeetCode 581)

Chỉ số lệch đầu tiên và cuối cùng xác định đoạn cần sắp; độ dài = `last - first + 1` (0 nếu không lệch).

In [24]:
def shortest_unsorted_length(nums: list[int]) -> int:
    idx = mismatched_indices(nums)
    return 0 if not idx else idx[-1] - idx[0] + 1

print(shortest_unsorted_length([2, 6, 4, 8, 10, 9, 15]))   # 5
print(shortest_unsorted_length([1, 2, 3]))                 # 0

5
0


## 16. Bài tập tự luyện (tăng dần)

**Bài 1 (tính tay):** `heights = [2,1,2,1,1,2,2,1]`. Viết `expected` và đếm số vị trí lệch.

**Bài 2 (điền chỗ trống):** hoàn thiện `ex_height_checker` bằng `sorted` + vòng `for`.

**Bài 3 (counting):** `ex_counting_sort(nums)` — trả về list đã sắp bằng bảng đếm, biết `0 <= nums[i] <= 100`, không dùng `sorted`.

**Bài 4 (biến thể):** `ex_height_checker_desc(heights)` — ảnh chụp yêu cầu đứng theo chiều cao **không tăng** (cao trước, thấp sau). Đếm vị trí lệch.

**Bài 5 (mini challenge):** `ex_smaller_counts(nums)` (LeetCode 1365) — với mỗi `nums[i]`, đếm có bao nhiêu số **nhỏ hơn hẳn** nó. Miền `0..100`, yêu cầu `O(n + K)`.

In [25]:
def ex_height_checker(heights: list[int]) -> int:
    # TODO 1: tạo expected bằng sorted (KHÔNG dùng heights.sort())
    # TODO 2: đếm số i mà heights[i] != expected[i]
    pass

# assert ex_height_checker([1, 1, 4, 2, 1, 3]) == 3
# assert ex_height_checker([2, 1, 2, 1, 1, 2, 2, 1]) == 4

In [26]:
def ex_counting_sort(nums: list[int]) -> list[int]:
    # TODO: freq = [0] * 101; đếm; trải lại theo v = 0..100
    pass

# assert ex_counting_sort([3, 0, 2, 3, 1]) == [0, 1, 2, 3, 3]

In [27]:
def ex_height_checker_desc(heights: list[int]) -> int:
    # TODO: thứ tự đúng là giảm dần
    pass

# assert ex_height_checker_desc([1, 2, 3]) == 2
# assert ex_height_checker_desc([3, 3, 1]) == 0

In [28]:
def ex_smaller_counts(nums: list[int]) -> list[int]:
    # TODO: đếm freq, cộng dồn (prefix sum) để biết có bao nhiêu số < v
    pass

# assert ex_smaller_counts([8, 1, 2, 2, 3]) == [4, 0, 1, 1, 3]

## 17. Gợi ý 3 mức

**Bài 1:** (1) Sắp `[2,1,2,1,1,2,2,1]`. (2) `expected = [1,1,1,1,2,2,2,2]`. (3) So từng vị trí: lệch ở 0, 2, 4, 7 → 4 vị trí.

**Bài 2:** (1) Giống cách 1. (2) `expected = sorted(heights)`. (3) `if heights[i] != expected[i]: count += 1`.

**Bài 3:** (1) Bảng đếm theo giá trị. (2) `for x in nums: freq[x] += 1`. (3) `for v in range(101): res.extend([v] * freq[v])`.

**Bài 4:** (1) Chỉ đổi thứ tự sắp. (2) `sorted(heights, reverse=True)`. (3) Sau đó đếm như cũ.

**Bài 5:** (1) "Số phần tử < v" = tổng `freq[0..v-1]`. (2) Tạo `less[v]` bằng cộng dồn. (3) `less[v+1] = less[v] + freq[v]`, trả `[less[x] for x in nums]`.

## 18. Đáp án tham khảo + template nhớ nhanh + checklist

Tự làm trước rồi mới xem. Đáp án bài 1: `4`.

In [29]:
def ans_height_checker(heights):
    expected = sorted(heights)
    count = 0
    for i in range(len(heights)):
        if heights[i] != expected[i]:
            count += 1
    return count

def ans_counting_sort(nums):
    freq = [0] * 101
    for x in nums:
        freq[x] += 1
    res = []
    for v in range(101):
        res.extend([v] * freq[v])
    return res

def ans_height_checker_desc(heights):
    expected = sorted(heights, reverse=True)
    return sum(h != e for h, e in zip(heights, expected))

def ans_smaller_counts(nums):
    freq = [0] * 101
    for x in nums:
        freq[x] += 1
    less = [0] * 102
    for v in range(101):
        less[v + 1] = less[v] + freq[v]
    return [less[x] for x in nums]

assert ans_height_checker([1, 1, 4, 2, 1, 3]) == 3
assert ans_height_checker([2, 1, 2, 1, 1, 2, 2, 1]) == 4
assert ans_counting_sort([3, 0, 2, 3, 1]) == [0, 1, 2, 3, 3]
assert ans_height_checker_desc([1, 2, 3]) == 2
assert ans_height_checker_desc([3, 3, 1]) == 0
assert ans_smaller_counts([8, 1, 2, 2, 3]) == [4, 0, 1, 1, 3]
assert ans_smaller_counts([7, 7, 7]) == [0, 0, 0]
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


### Template nhớ nhanh

```python
class Solution:
    def heightChecker(self, heights: list[int]) -> int:
        expected = sorted(heights)                       # bản sao đã sắp
        return sum(h != e for h, e in zip(heights, expected))
```

Counting sort khi miền giá trị nhỏ: `freq = [0] * (MAX + 1)` → đếm → trải lại theo thứ tự giá trị.

### Checklist trước khi submit
- [ ] Dùng `sorted(heights)` (bản sao), **không** `heights.sort()` rồi so với chính nó.
- [ ] Không gán `x = heights.sort()` (sẽ là `None`).
- [ ] Đếm **số vị trí lệch**, không phải số lần đổi chỗ.
- [ ] Nếu đếm bằng mảng: kích thước `101` để có chỉ số 100.
- [ ] Test `[5,1,2,3,4]` → 5 và `[1,2,3,4,5]` → 0.
- [ ] Độ phức tạp: sort `O(n log n)`; counting `O(n + K)`.